<h1>Chapter 1 - Introduction to Language Models</h1>
<i>Exploring the exciting field of Language AI</i>


<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TUSIDENG/Hands-On-Large-Language-Models/blob/main/chapter01/Chapter%201%20-%20Introduction%20to%20Language%20Models.ipynb)

---

This notebook is for Chapter 1 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

Jupyter Notebook 的单元格魔法命令（Magic Command）Jupyter Notebook 的单元格魔法命令（Magic Command）：
- %% 表示作用于整个单元格（区别于%仅作用于单行）；
- capture 意为 “捕获 / 拦截”，作用是隐藏该单元格内所有命令的输出日志（包括 pip 安装时的下载进度、依赖解析、成功 / 警告信息等）；
- 开头的#是 Jupyter 魔法命令的固定格式，不是普通注释，删除后该命令失效。

执行系统级 pip 安装命令：
- ! 是 Jupyter 调用系统终端命令的前缀；
- 安装transformers 4.41.2（大模型调用核心库）和accelerate 0.31.0（模型加速库），并锁定版本避免兼容问题。

安装 transformers 库的4.41.2 版本（而非最新版）：
- transformers 是 Hugging Face 开源的核心库，封装了几乎所有主流大模型（如 GPT、Llama、Qwen）的预训练 / 推理代码，能快速调用模型、处理 Token、加载权重；
- ==4.41.2 是 “版本锁定”，避免自动升级到新版导致代码兼容问题（比如新版 API 变更会让旧代码报错）。

安装 accelerate 库的0.31.0 版本：
- accelerate 同样是 Hugging Face 的库，核心作用是优化大模型的训练 / 推理效率，自动适配 GPU/CPU/ 多卡集群，简化分布式训练、混合精度计算等复杂配置（比如让普通显卡也能高效运行大模型）；
- 锁定版本是为了和transformers==4.41.2 匹配，避免版本不兼容（比如新版 accelerate 可能不支持旧版 transformers）。

In [ ]:
# %%capture
# !pip install transformers==4.41.2 accelerate==0.31.0

# Phi-3

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately (although that isn't always necessary).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
# 安全、高效地加载微软 Phi-3-mini-4k-instruct 轻量级大模型到 GPU（CUDA），并匹配对应的分词器。
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

Although we can now use the model and tokenizer directly, it's much easier to wrap it in a `pipeline` object:

In [ ]:
from transformers import pipeline

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

Finally, we create our prompt as a user and give it to the model:

In [ ]:
# The prompt (user input / query)
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

# Generate output
output = generator(messages)
print(output[0]["generated_text"])